# pix2db — image → working SQLite database

Upload a photo or screenshot of a table, form, invoice, receipt or chart and get a real SQLite database back: schema with types and keys, all rows, and a verify/repair report.

**Runtime → Change runtime type → T4 GPU** (free), then run the cells top to bottom.

In [ ]:
!pip -q install "git+https://github.com/m-peker/pix2db"

In [ ]:
import torch
from pix2db import Pix2DB

# T4 has no bfloat16: use float16 there
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = Pix2DB.from_pretrained("musapeker/pix2db-2b", dtype=dtype)

## Upload an image

In [ ]:
from google.colab import files
from PIL import Image
import io

up = files.upload()
image = Image.open(io.BytesIO(next(iter(up.values())))).convert("RGB")
image

In [ ]:
db = model.extract(image)
print(db.sql)
print("
repairs:", db.repairs or "none")
print("warnings:", db.warnings or "none")

## The tables

In [ ]:
from IPython.display import display
for name, df in db.tables.items():
    print(name)
    display(df)

## Query it with SQL

In [ ]:
first = next(iter(db.tables))
db.query(f'SELECT * FROM "{first}" LIMIT 5')

In [ ]:
db.save("extracted.sqlite")
files.download("extracted.sqlite")